## Implémentation de l'attention à tête unique

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SingleHeadAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim

        # Linear projections for Query, Key, Value
        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.value_proj = nn.Linear(hidden_dim, hidden_dim)

        # Scaling factor
        self.scale = torch.sqrt(torch.tensor(hidden_dim, dtype=torch.float32))

    def forward(self, query, key, value, mask=None):
        # Project inputs
        Q = self.query_proj(query) # (batch_size, seq_len, hidden_dim)
        K = self.key_proj(key)     # (batch_size, seq_len, hidden_dim)
        V = self.value_proj(value) # (batch_size, seq_len, hidden_dim)

        # Calculate attention scores: QK^T
        # (batch_size, seq_len, hidden_dim) @ (batch_size, hidden_dim, seq_len) -> (batch_size, seq_len, seq_len)
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale

        # Apply mask if provided
        if mask is not None:
            attention_scores = attention_scores.masked_fill(mask == 0, -1e9) # Fill with a very small number

        # Apply softmax to get attention weights
        attention_weights = F.softmax(attention_scores, dim=-1)

        # Multiply weights by Value
        # (batch_size, seq_len, seq_len) @ (batch_size, seq_len, hidden_dim) -> (batch_size, seq_len, hidden_dim)
        output = torch.matmul(attention_weights, V)

        return output, attention_weights


In [2]:
print("Validating SingleHeadAttention module with dummy tensors...")

batch_size = 2
seq_len = 5
hidden_dim = 64

# Create dummy input tensors
dummy_input = torch.randn(batch_size, seq_len, hidden_dim)

# Instantiate the module
single_head_attention_layer = SingleHeadAttention(hidden_dim)

# Forward pass
output, attention_weights = single_head_attention_layer(dummy_input, dummy_input, dummy_input)

# Print shapes to validate
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attention_weights.shape}")

# Expected shapes:
# Input shape: (batch_size, seq_len, hidden_dim) -> (2, 5, 64)
# Output shape: (batch_size, seq_len, hidden_dim) -> (2, 5, 64)
# Attention weights shape: (batch_size, seq_len, seq_len) -> (2, 5, 5)

# Assertions for programmatic check
assert output.shape == torch.Size([batch_size, seq_len, hidden_dim]), "Output shape mismatch!"
assert attention_weights.shape == torch.Size([batch_size, seq_len, seq_len]), "Attention weights shape mismatch!"

print("Shape validation successful!")
print("Example attention weights (first batch, first token):\n", attention_weights[0, 0])


Validating SingleHeadAttention module with dummy tensors...
Input shape: torch.Size([2, 5, 64])
Output shape: torch.Size([2, 5, 64])
Attention weights shape: torch.Size([2, 5, 5])
Shape validation successful!
Example attention weights (first batch, first token):
 tensor([0.2886, 0.1969, 0.2195, 0.1517, 0.1433], grad_fn=<SelectBackward0>)


## Module d'attention multi-têtes

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout_rate=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        if hidden_dim % num_heads != 0:
            raise ValueError("hidden_dim must be divisible by num_heads")

        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.value_proj = nn.Linear(hidden_dim, hidden_dim)
        self.output_proj = nn.Linear(hidden_dim, hidden_dim)

        self.dropout = nn.Dropout(dropout_rate)
        self.scale = torch.sqrt(torch.tensor(self.head_dim, dtype=torch.float32))

    def forward(self, query, key, value, mask=None):
        batch_size = query.shape[0]

        # Project and reshape (batch_size, seq_len, hidden_dim) -> (batch_size, seq_len, num_heads, head_dim)
        # Then transpose to (batch_size, num_heads, seq_len, head_dim)
        Q = self.query_proj(query).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.key_proj(key).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.value_proj(value).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        # Calculate attention scores: (batch_size, num_heads, seq_len, head_dim) @ (batch_size, num_heads, head_dim, seq_len)
        # -> (batch_size, num_heads, seq_len, seq_len)
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale

        # Apply mask if provided.
        # Incoming mask from EncoderModel/CustomEncoder will be (batch_size, seq_len)
        # We need to expand it to (batch_size, 1, 1, seq_len) for broadcasting over num_heads and query_seq_len
        if mask is not None:
            expanded_mask = mask.unsqueeze(1).unsqueeze(2) # -> (batch_size, 1, 1, seq_len)
            attention_scores = attention_scores.masked_fill(expanded_mask == 0, -1e9)

        # Apply softmax to get attention weights
        attention_weights = F.softmax(attention_scores, dim=-1)

        # Apply dropout to attention weights
        attention_weights = self.dropout(attention_weights)

        # Multiply weights by Value: (batch_size, num_heads, seq_len, seq_len) @ (batch_size, num_heads, seq_len, head_dim)
        # -> (batch_size, num_heads, seq_len, head_dim)
        x = torch.matmul(attention_weights, V)

        # Concatenate heads and project back to original hidden_dim
        # (batch_size, num_heads, seq_len, head_dim) -> (batch_size, seq_len, num_heads, head_dim) -> (batch_size, seq_len, hidden_dim)
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.hidden_dim)

        output = self.output_proj(x)

        return output, attention_weights


class MultiHeadAttentionBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout_rate=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(hidden_dim, num_heads, dropout_rate)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.dropout1 = nn.Dropout(dropout_rate)

        # Feedforward layer (often called Position-wise Feed-Forward Network)
        self.feed_forward = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim), # Typically 4*hidden_dim for intermediate size
            nn.ReLU(),
            nn.Linear(4 * hidden_dim, hidden_dim)
        )
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.dropout2 = nn.Dropout(dropout_rate)

    def forward(self, x, mask=None):
        # Multi-Head Attention with Skip Connection and LayerNorm
        _x, attn_weights = self.attention(x, x, x, mask) # Self-attention
        x = self.norm1(x + self.dropout1(_x)) # Add & Norm

        # Feed-Forward with Skip Connection and LayerNorm
        _x = self.feed_forward(x)
        x = self.norm2(x + self.dropout2(_x)) # Add & Norm

        return x, attn_weights # Return attention weights as well


In [4]:
print("Validating MultiHeadAttention module with dummy tensors...")

batch_size = 2
seq_len = 10
hidden_dim = 256 # Must be divisible by num_heads
num_heads = 8
dropout_rate = 0.1

# Create dummy input tensors
dummy_input = torch.randn(batch_size, seq_len, hidden_dim)

# Instantiate the MultiHeadAttention module
multi_head_attention_layer = MultiHeadAttention(hidden_dim, num_heads, dropout_rate)

# Forward pass
output_mha, attention_weights_mha = multi_head_attention_layer(dummy_input, dummy_input, dummy_input)

# Print shapes to validate
print(f"MultiHeadAttention Input shape: {dummy_input.shape}")
print(f"MultiHeadAttention Output shape: {output_mha.shape}")
print(f"MultiHeadAttention Weights shape: {attention_weights_mha.shape}")

# Expected shapes:
# Input shape: (batch_size, seq_len, hidden_dim) -> (2, 10, 256)
# Output shape: (batch_size, seq_len, hidden_dim) -> (2, 10, 256)
# Attention weights shape: (batch_size, num_heads, seq_len, seq_len) -> (2, 8, 10, 10)

# Assertions for programmatic check
assert output_mha.shape == torch.Size([batch_size, seq_len, hidden_dim]), "MultiHeadAttention Output shape mismatch!"
assert attention_weights_mha.shape == torch.Size([batch_size, num_heads, seq_len, seq_len]), "MultiHeadAttention Weights shape mismatch!"

print("MultiHeadAttention shape validation successful!")

print("\nValidating MultiHeadAttentionBlock module with dummy tensors...")
# Instantiate the MultiHeadAttentionBlock module
multi_head_attention_block = MultiHeadAttentionBlock(hidden_dim, num_heads, dropout_rate)

# Forward pass
output_block, attn_weights_block = multi_head_attention_block(dummy_input)

# Print shapes to validate
print(f"MultiHeadAttentionBlock Input shape: {dummy_input.shape}")
print(f"MultiHeadAttentionBlock Output shape: {output_block.shape}")
print(f"MultiHeadAttentionBlock Attention Weights shape: {attn_weights_block.shape}")

# Expected shapes:
# Input shape: (batch_size, seq_len, hidden_dim) -> (2, 10, 256)
# Output shape: (batch_size, seq_len, hidden_dim) -> (2, 10, 256)
# Attention weights shape: (batch_size, num_heads, seq_len, seq_len) -> (2, 8, 10, 10)

# Assertions for programmatic check
assert output_block.shape == torch.Size([batch_size, seq_len, hidden_dim]), "MultiHeadAttentionBlock Output shape mismatch!"
assert attn_weights_block.shape == torch.Size([batch_size, num_heads, seq_len, seq_len]), "MultiHeadAttentionBlock Attention Weights shape mismatch!"

print("MultiHeadAttentionBlock shape validation successful!")
print("Example output (first batch, first token, first 5 hidden_dim values):\n", output_block[0, 0, :5])


Validating MultiHeadAttention module with dummy tensors...
MultiHeadAttention Input shape: torch.Size([2, 10, 256])
MultiHeadAttention Output shape: torch.Size([2, 10, 256])
MultiHeadAttention Weights shape: torch.Size([2, 8, 10, 10])
MultiHeadAttention shape validation successful!

Validating MultiHeadAttentionBlock module with dummy tensors...
MultiHeadAttentionBlock Input shape: torch.Size([2, 10, 256])
MultiHeadAttentionBlock Output shape: torch.Size([2, 10, 256])
MultiHeadAttentionBlock shape validation successful!
Example output (first batch, first token, first 5 hidden_dim values):
 tensor([ 1.0699, -1.4996, -1.3131,  0.6629, -1.7700], grad_fn=<SliceBackward0>)


## Pile d'encodeurs personnalisés & Boucle d'entraînement (Optionnel)

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Re-using MultiHeadAttentionBlock defined earlier
# class MultiHeadAttention(nn.Module): ...
# class MultiHeadAttentionBlock(nn.Module): ...

class CustomEncoder(nn.Module):
    def __init__(self, hidden_dim, num_heads, num_layers, dropout_rate=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            MultiHeadAttentionBlock(hidden_dim, num_heads, dropout_rate)
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        all_attention_weights = []
        for layer in self.layers:
            x, attn_weights = layer(x, mask)
            all_attention_weights.append(attn_weights)
        return x, all_attention_weights

class EncoderModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_heads, num_layers, num_classes, dropout_rate=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.encoder = CustomEncoder(hidden_dim, num_heads, num_layers, dropout_rate)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, input_ids, mask=None):
        # input_ids: (batch_size, seq_len)
        embedded = self.dropout(self.embedding(input_ids))

        # Pass through the custom encoder stack
        # Output of encoder: (batch_size, seq_len, hidden_dim)
        encoded_output, all_attention_weights = self.encoder(embedded, mask)

        # For classification, typically take the output of the first token (CLS token equivalent)
        # or average pooling. Here, we'll take the first token.
        cls_output = encoded_output[:, 0, :]

        # Final linear layer for classification
        logits = self.fc(self.dropout(cls_output))

        return logits, all_attention_weights


In [13]:
print("Validating CustomEncoder and EncoderModel with dummy tensors (with attention weights)...")

batch_size = 2
seq_len = 10
hidden_dim = 256
num_heads = 8
num_layers = 2
vocab_size = 10000 # Example vocab size
num_classes = 3    # For NLI (entailment, neutral, contradiction)
dropout_rate = 0.1

# Create dummy input_ids and mask
dummy_input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
# Dummy mask (all ones for now, meaning no padding)
dummy_mask_input = torch.ones(batch_size, seq_len, dtype=torch.bool) # Simplified mask for illustration

# Instantiate the EncoderModel
model = EncoderModel(vocab_size, hidden_dim, num_heads, num_layers, num_classes, dropout_rate)

# Forward pass
logits, all_attention_weights = model(dummy_input_ids, dummy_mask_input)

# Print shapes to validate
print(f"Dummy input_ids shape: {dummy_input_ids.shape}")
print(f"Dummy mask_input shape: {dummy_mask_input.shape}")
print(f"Model output (logits) shape: {logits.shape}")
print(f"Number of attention weight layers: {len(all_attention_weights)}")
print(f"Shape of attention weights for first layer: {all_attention_weights[0].shape}")

# Expected shapes:
# input_ids: (batch_size, seq_len) -> (2, 10)
# mask: (batch_size, seq_len) -> (2, 10)
# logits: (batch_size, num_classes) -> (2, 3)
# all_attention_weights: List of num_layers tensors, each of shape (batch_size, num_heads, seq_len, seq_len)

# Assertions for programmatic check
assert logits.shape == torch.Size([batch_size, num_classes]), "EncoderModel output (logits) shape mismatch!"
assert len(all_attention_weights) == num_layers, "Number of attention weight layers mismatch!"
assert all_attention_weights[0].shape == torch.Size([batch_size, num_heads, seq_len, seq_len]), "Attention weights shape mismatch!"

print("CustomEncoder and EncoderModel (with attention weights) shape validation successful!")
print("Example logits (first batch):\n", logits[0])
print("Example attention weights (first batch, first layer, first head, first token):\n", all_attention_weights[0][0, 0, 0])


Validating CustomEncoder and EncoderModel with dummy tensors (with attention weights)...
Dummy input_ids shape: torch.Size([2, 10])
Dummy mask_input shape: torch.Size([2, 10])
Model output (logits) shape: torch.Size([2, 3])
Number of attention weight layers: 2
Shape of attention weights for first layer: torch.Size([2, 8, 10, 10])
CustomEncoder and EncoderModel (with attention weights) shape validation successful!
Example logits (first batch):
 tensor([0.0722, 0.1197, 0.4589], grad_fn=<SelectBackward0>)
Example attention weights (first batch, first layer, first head, first token):
 tensor([0.0747, 0.0000, 0.1442, 0.2141, 0.0495, 0.1561, 0.0710, 0.0621, 0.0888,
        0.1586], grad_fn=<SelectBackward0>)


### Placeholder pour la tokenisation du jeu de données et la boucle d'entraînement

Pour la mise en œuvre complète, vous auriez besoin de:

1.  **Charger le jeu de données NLI**: Utiliser `datasets` ou charger manuellement les fichiers JSONL/CSV.
2.  **Tokenisation**: Utiliser un tokenizer comme `AutoTokenizer` de Hugging Face (`DistilBertTokenizer`, `BertTokenizer`) pour convertir les prémisses et hypothèses en `input_ids` et créer des masques d'attention.
3.  **DataLoader**: Créer des `DataLoader` PyTorch pour les lots d'entraînement et de validation.
4.  **Boucle d'entraînement**: Implémenter une boucle pour l'entraînement sur plusieurs époques, le calcul de la perte, l'optimisation (e.g., `torch.optim.AdamW`), et l'évaluation sur le jeu de validation.

```python
# Exemple conceptuel de la boucle d'entraînement (non exécutable sans le jeu de données et le tokenizer)

# import datasets # pip install datasets
# from transformers import AutoTokenizer # pip install transformers
# from torch.utils.data import DataLoader, Dataset
# from tqdm.auto import tqdm # pip install tqdm

# # 1. Charger le jeu de données NLI (exemple conceptuel)
# # dataset = datasets.load_dataset("multi_nli") # ou charger votre dataset local

# # 2. Tokenisation (exemple conceptuel)
# # tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# # def tokenize_function(examples):
# #     # Concaténer prémisse et hypothèse pour le NLI
# #     return tokenizer(examples["premise"], examples["hypothesis"], truncation=True, padding="max_length")
# # tokenized_datasets = dataset.map(tokenize_function, batched=True)
# # tokenized_datasets = tokenized_datasets.remove_columns(["premise", "hypothesis", "label"])
# # tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
# # tokenized_datasets.set_format("torch")

# # 3. DataLoaders (exemple conceptuel)
# # train_dataloader = DataLoader(tokenized_datasets["train"], shuffle=True, batch_size=16)
# # eval_dataloader = DataLoader(tokenized_datasets["validation_matched"], batch_size=16)

# # 4. Boucle d'entraînement (exemple conceptuel)
# # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# # model = EncoderModel(vocab_size=tokenizer.vocab_size, hidden_dim=hidden_dim, num_heads=num_heads, num_layers=num_layers, num_classes=num_classes).to(device)
# # optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
# # criterion = nn.CrossEntropyLoss()

# # num_epochs = 3
# # for epoch in range(num_epochs):
# #     model.train()
# #     total_loss = 0
# #     for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1} Training"):
# #         input_ids = batch["input_ids"].to(device)
# #         attention_mask = batch["attention_mask"].to(device)
# #         labels = batch["labels"].to(device)
        
# #         optimizer.zero_grad()
# #         logits, _ = model(input_ids, attention_mask) # Now returns attention weights
# #         loss = criterion(logits, labels)
# #         total_loss += loss.item()
# #         loss.backward()
# #         optimizer.step()
# #     print(f"Epoch {epoch+1} Average Loss: {total_loss / len(train_dataloader)}")
    
# #     # Validation loop (simplified)
# #     model.eval()
# #     correct_predictions = 0
# #     total_samples = 0
# #     with torch.no_grad():
# #         for batch in tqdm(eval_dataloader, desc=f"Epoch {epoch+1} Validation"):
# #             input_ids = batch["input_ids"].to(device)
# #             attention_mask = batch["attention_mask"].to(device)
# #             labels = batch["labels"].to(device)
            
# #             logits, _ = model(input_ids, attention_mask) # Now returns attention weights
# #             predictions = torch.argmax(logits, dim=-1)
# #             correct_predictions += (predictions == labels).sum().item()
# #             total_samples += labels.size(0)
# #     print(f"Epoch {epoch+1} Validation Accuracy: {correct_predictions / total_samples}")

```
### Extraction des poids d'attention de DistilBERT

Pour extraire les poids d'attention d'un modèle Hugging Face comme DistilBERT, vous devez appeler la méthode `forward` avec `output_attentions=True`. Cela renverra un tuple, où le premier élément est la sortie normale du modèle et le deuxième est un tuple contenant les poids d'attention pour chaque couche. Les poids d'attention seront de la forme `(batch_size, num_heads, seq_len, seq_len)`.


In [14]:
print("Demonstrating attention weights extraction from DistilBERT...")

# Re-load model with output_attentions=True
model_distilbert_attn = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_nli_classes,
    output_attentions=True # Crucial for getting attention weights
)

# Ensure it's in evaluation mode
model_distilbert_attn.eval()

# Use the same dummy input from previous step
# dummy_input_ids_distilbert and dummy_attention_mask_distilbert are already defined

with torch.no_grad():
    outputs_distilbert_attn = model_distilbert_attn(
        input_ids=dummy_input_ids_distilbert,
        attention_mask=dummy_attention_mask_distilbert
    )

logits_distilbert = outputs_distilbert_attn.logits
attention_weights_distilbert = outputs_distilbert_attn.attentions # This will be a tuple of tensors

print(f"DistilBERT model output (logits) shape: {logits_distilbert.shape}")
print(f"Number of attention weight layers (DistilBERT): {len(attention_weights_distilbert)}")
print(f"Shape of attention weights for first layer (DistilBERT): {attention_weights_distilbert[0].shape}")

# Expected shape for DistilBERT attention weights:
# A tuple of `num_hidden_layers` tensors, each of shape (batch_size, num_heads, seq_len, seq_len)
# DistilBERT has 6 hidden layers (num_layers), 12 heads, and max_seq_len (here 50)

num_distilbert_layers = model_distilbert_attn.config.num_hidden_layers
num_distilbert_heads = model_distilbert_attn.config.num_attention_heads

assert len(attention_weights_distilbert) == num_distilbert_layers, "DistilBERT: Number of attention weight layers mismatch!"
assert attention_weights_distilbert[0].shape == torch.Size([1, num_distilbert_heads, max_seq_len, max_seq_len]), "DistilBERT: Attention weights shape mismatch!"

print("DistilBERT attention weights extraction successful!")
print("Example DistilBERT attention weights (first batch, first layer, first head, first token):\n", attention_weights_distilbert[0][0, 0, 0, :])


Demonstrating attention weights extraction from DistilBERT...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT model output (logits) shape: torch.Size([1, 3])
Number of attention weight layers (DistilBERT): 6
Shape of attention weights for first layer (DistilBERT): torch.Size([1, 12, 50, 50])
DistilBERT attention weights extraction successful!
Example DistilBERT attention weights (first batch, first layer, first head, first token):
 tensor([0.0328, 0.0446, 0.0235, 0.0535, 0.0203, 0.0552, 0.1298, 0.1450, 0.0537,
        0.0260, 0.0595, 0.0234, 0.0588, 0.1245, 0.1494, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000])


## Visualisation des cartes d'attention et réflexion

Pour analyser les cartes d'attention et interpréter la mise au point du modèle, vous pouvez utiliser des bibliothèques comme `matplotlib` ou `seaborn` pour visualiser les matrices d'attention.

### Étapes pour la visualisation et l'analyse:

1.  **Préparer les données**: Obtenez les poids d'attention pour un échantillon spécifique de prémisse/hypothèse à partir de votre `EncoderModel` personnalisé et du `DistilBERT`.
2.  **Visualiser**: Pour chaque couche et chaque tête, tracez la matrice d'attention `(seq_len, seq_len)`. Vous pouvez utiliser des cartes thermiques pour montrer l'intensité de l'attention entre les tokens. Pour une meilleure interprétation, vous devrez également récupérer les tokens réels.
3.  **Interpréter**: Observez quels tokens prêtent attention à quels autres tokens. Cherchez des modèles, par exemple, si les tokens se concentrent sur des mots-clés, des pronoms, des entités spécifiques, ou des relations entre la prémisse et l'hypothèse.
4.  **Comparaison**: Comparez les cartes d'attention de votre modèle personnalisé et de DistilBERT. Votre modèle personnalisé se concentre-t-il sur des aspects similaires ou différents du texte? Un modèle pré-entraîné peut avoir appris des modèles d'attention plus sophistiqués ou généralisables.
5.  **Réflexion sur les compromis**: Discutez des compromis entre un modèle personnalisé léger et un grand modèle pré-entraîné:
    *   **Performance**: Un modèle pré-entraîné (comme DistilBERT) aura probablement une meilleure performance globale sur des tâches NLI en raison de son entraînement sur des quantités massives de données et de sa capacité à capturer des représentations linguistiques complexes. Votre modèle personnalisé, étant plus léger, pourrait avoir besoin de beaucoup plus de données spécifiques à la tâche et d'un réglage fin pour atteindre des performances comparables.
    *   **Complexité et ressources**: Le modèle personnalisé est plus simple et consomme moins de ressources (mémoire, calcul) à l'entraînement et à l'inférence. DistilBERT est plus grand et plus gourmand en ressources.
    *   **Interprétabilité**: Les modèles plus simples peuvent parfois être plus faciles à interpréter. Cependant, les cartes d'attention peuvent fournir des informations précieuses pour les deux.
    *   **Transfert d'apprentissage**: Les modèles pré-entraînés excellent dans le transfert d'apprentissage, c'est-à-dire qu'ils peuvent être finement réglés sur de petits ensembles de données spécifiques à la tâche avec de bons résultats.

```python
import matplotlib.pyplot as plt
import seaborn as sns

def plot_attention_weights(attention_weights, tokens, title="Attention Map", layer_idx=0, head_idx=0):
    """Plots the attention weights for a specific layer and head."""
    if isinstance(attention_weights, tuple): # For HuggingFace models, it's a tuple
        attn_matrix = attention_weights[layer_idx][0, head_idx].cpu().numpy() # Assuming batch_size=1
    else: # For our custom model, it's a list of tensors
        attn_matrix = attention_weights[layer_idx][0, head_idx].cpu().numpy() # Assuming batch_size=1
        
    plt.figure(figsize=(10, 8))
    sns.heatmap(attn_matrix, xticklabels=tokens, yticklabels=tokens, cmap="viridis")
    plt.title(f"{title} - Layer {layer_idx+1}, Head {head_idx+1}")
    plt.xlabel("Key (Tokens)")
    plt.ylabel("Query (Tokens)")
    plt.tight_layout()
    plt.show()

# Example usage (conceptual - requires actual tokenized input and model outputs):
# tokens_custom_model = ["[CLS]", "token1", "token2", ..., "[SEP]"]
# tokens_distilbert = tokenizer_distilbert.convert_ids_to_tokens(dummy_input_ids_distilbert[0].tolist())

# # Assuming you have run forward passes and got attention weights:
# # logits_custom, all_attention_weights_custom = model(input_ids, attention_mask)
# # outputs_distilbert_attn = model_distilbert_attn(input_ids=input_ids_distilbert, attention_mask=attention_mask_distilbert)
# # attention_weights_distilbert = outputs_distilbert_attn.attentions

# if 'all_attention_weights_custom' in locals() and 'tokens_custom_model' in locals():
#     print("\nVisualizing Custom Encoder Attention (Layer 1, Head 1):")
#     plot_attention_weights(all_attention_weights_custom, tokens_custom_model, "Custom Encoder Attention", layer_idx=0, head_idx=0)

# if 'attention_weights_distilbert' in locals() and 'tokens_distilbert' in locals():
#     print("\nVisualizing DistilBERT Attention (Layer 1, Head 1):")
#     plot_attention_weights(attention_weights_distilbert, tokens_distilbert, "DistilBERT Attention", layer_idx=0, head_idx=0)

# Reflection: This section would contain your written analysis comparing the attention patterns,
# model performance, and resource usage between the custom model and DistilBERT.
# You would discuss which tokens received more attention, if the patterns make sense for NLI,
# and the trade-offs of using a simpler custom model versus a powerful pre-trained one.
```


## Comparaison avec un transformateur pré-entraîné (DistilBERT)

In [7]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Loading DistilBERT tokenizer and model for sequence classification...")

# 1. Load pre-trained tokenizer
tokenizer_distilbert = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# 2. Load pre-trained model with a classification head
# For NLI, we typically have 3 classes: entailment, neutral, contradiction
num_nli_classes = 3
model_distilbert = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_nli_classes
)

print("DistilBERT tokenizer and model loaded successfully.")
print(f"DistilBERT model architecture: {model_distilbert.config}")

# 3. Validate with dummy inputs
print("\nValidating DistilBERT model with dummy tensors...")

max_seq_len = 50 # A typical max sequence length for Transformer models

# Create dummy input text and tokenize it
dummy_text_premise = "This is a dummy premise."
dummy_text_hypothesis = "This is a dummy hypothesis."

encoded_input = tokenizer_distilbert(
    dummy_text_premise,
    dummy_text_hypothesis,
    padding="max_length",
    truncation=True,
    max_length=max_seq_len,
    return_tensors="pt"
)

# Get input_ids and attention_mask
dummy_input_ids_distilbert = encoded_input["input_ids"]
dummy_attention_mask_distilbert = encoded_input["attention_mask"]

print(f"Dummy input_ids shape (DistilBERT): {dummy_input_ids_distilbert.shape}")
print(f"Dummy attention_mask shape (DistilBERT): {dummy_attention_mask_distilbert.shape}")

# Forward pass through the DistilBERT model
model_distilbert.eval() # Set to evaluation mode
with torch.no_grad():
    outputs_distilbert = model_distilbert(
        input_ids=dummy_input_ids_distilbert,
        attention_mask=dummy_attention_mask_distilbert
    )

logits_distilbert = outputs_distilbert.logits

print(f"DistilBERT model output (logits) shape: {logits_distilbert.shape}")

# Expected shape: (batch_size, num_nli_classes) -> (1, 3) for our dummy single input
assert logits_distilbert.shape == torch.Size([1, num_nli_classes]), "DistilBERT output shape mismatch!"

print("DistilBERT model shape validation successful!")
print("Example logits (DistilBERT):\n", logits_distilbert)


Loading DistilBERT tokenizer and model for sequence classification...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT tokenizer and model loaded successfully.
DistilBERT model architecture: DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "initializer_range": 0.02,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.12.0",
  "vocab_size": 30522
}


Validating DistilBERT model with dummy tensors...
Dummy input_ids shape (DistilBERT): torch.Size([1, 50])
Dummy attention_mask shape (DistilBERT): torch.Size([1, 50]

## Visualisation des cartes d'attention et réflexion

Pour analyser les cartes d'attention et interpréter la mise au point du modèle, vous pouvez utiliser des bibliothèques comme `matplotlib` ou `seaborn` pour visualiser les matrices d'attention.

### Étapes pour la visualisation et l'analyse:

1.  **Préparer les données**: Obtenez les poids d'attention pour un échantillon spécifique de prémisse/hypothèse à partir de votre `EncoderModel` personnalisé et du `DistilBERT`.
2.  **Visualiser**: Pour chaque couche et chaque tête, tracez la matrice d'attention `(seq_len, seq_len)`. Vous pouvez utiliser des cartes thermiques pour montrer l'intensité de l'attention entre les tokens. Pour une meilleure interprétation, vous devrez également récupérer les tokens réels.
3.  **Interpréter**: Observez quels tokens prêtent attention à quels autres tokens. Cherchez des modèles, par exemple, si les tokens se concentrent sur des mots-clés, des pronoms, des entités spécifiques, ou des relations entre la prémisse et l'hypothèse.
4.  **Comparaison**: Comparez les cartes d'attention de votre modèle personnalisé et de DistilBERT. Votre modèle personnalisé se concentre-t-il sur des aspects similaires ou différents du texte? Un modèle pré-entraîné peut avoir appris des modèles d'attention plus sophistiqués ou généralisables.
5.  **Réflexion sur les compromis**: Discutez des compromis entre un modèle personnalisé léger et un grand modèle pré-entraîné:
    *   **Performance**: Un modèle pré-entraîné (comme DistilBERT) aura probablement une meilleure performance globale sur des tâches NLI en raison de son entraînement sur des quantités massives de données et de sa capacité à capturer des représentations linguistiques complexes. Votre modèle personnalisé, étant plus léger, pourrait avoir besoin de beaucoup plus de données spécifiques à la tâche et d'un réglage fin pour atteindre des performances comparables.
    *   **Complexité et ressources**: Le modèle personnalisé est plus simple et consomme moins de ressources (mémoire, calcul) à l'entraînement et à l'inférence. DistilBERT est plus grand et plus gourmand en ressources.
    *   **Interprétabilité**: Les modèles plus simples peuvent parfois être plus faciles à interpréter. Cependant, les cartes d'attention peuvent fournir des informations précieuses pour les deux.
    *   **Transfert d'apprentissage**: Les modèles pré-entraînés excellent dans le transfert d'apprentissage, c'est-à-dire qu'ils peuvent être finement réglés sur de petits ensembles de données spécifiques à la tâche avec de bons résultats.

```python
import matplotlib.pyplot as plt
import seaborn as sns

def plot_attention_weights(attention_weights, tokens, title="Attention Map", layer_idx=0, head_idx=0):
    """Plots the attention weights for a specific layer and head."""
    if isinstance(attention_weights, tuple): # For HuggingFace models, it's a tuple
        attn_matrix = attention_weights[layer_idx][0, head_idx].cpu().numpy() # Assuming batch_size=1
    else: # For our custom model, it's a list of tensors
        attn_matrix = attention_weights[layer_idx][0, head_idx].cpu().numpy() # Assuming batch_size=1

    plt.figure(figsize=(10, 8))
    sns.heatmap(attn_matrix, xticklabels=tokens, yticklabels=tokens, cmap="viridis")
    plt.title(f"{title} - Layer {layer_idx+1}, Head {head_idx+1}")
    plt.xlabel("Key (Tokens)")
    plt.ylabel("Query (Tokens)")
    plt.tight_layout()
    plt.show()

# Example usage (conceptual - requires actual tokenized input and model outputs):
# tokens_custom_model = ["[CLS]", "token1", "token2", ..., "[SEP]"]
# tokens_distilbert = tokenizer_distilbert.convert_ids_to_tokens(dummy_input_ids_distilbert[0].tolist())

# # Assuming you have run forward passes and got attention weights:
# # logits_custom, all_attention_weights_custom = model(input_ids, attention_mask)
# # outputs_distilbert_attn = model_distilbert_attn(input_ids=input_ids_distilbert, attention_mask=attention_mask_distilbert)
# # attention_weights_distilbert = outputs_distilbert_attn.attentions

# if 'all_attention_weights_custom' in locals() and 'tokens_custom_model' in locals():
#     print("\nVisualizing Custom Encoder Attention (Layer 1, Head 1):")
#     plot_attention_weights(all_attention_weights_custom, tokens_custom_model, "Custom Encoder Attention", layer_idx=0, head_idx=0)

# if 'attention_weights_distilbert' in locals() and 'tokens_distilbert' in locals():
#     print("\nVisualizing DistilBERT Attention (Layer 1, Head 1):")
#     plot_attention_weights(attention_weights_distilbert, tokens_distilbert, "DistilBERT Attention", layer_idx=0, head_idx=0)

# Reflection: This section would contain your written analysis comparing the attention patterns,
# model performance, and resource usage between the custom model and DistilBERT.
# You would discuss which tokens received more attention, if the patterns make sense for NLI,
# and the trade-offs of using a simpler custom model versus a powerful pre-trained one.
```


## Différences entre l'attention à tête unique, multi-têtes et croisée

### Attention à tête unique (Single-Head Attention)

L'attention à tête unique, comme celle que nous avons implémentée dans la classe `SingleHeadAttention`, est la forme fondamentale du mécanisme d'attention. Pour chaque token de la séquence d'entrée, elle calcule un score de pertinence par rapport à chaque autre token de la séquence. Ces scores sont ensuite normalisés (généralement via softmax) pour obtenir des poids, qui sont utilisés pour créer une somme pondérée des valeurs. Il n'y a qu'un seul ensemble de matrices de projection (Query, Key, Value).

*   **Mécanisme**: Calcule une seule distribution de poids d'attention pour chaque Query, agrège les Values en fonction de ces poids.
*   **Vue**: Chaque token se concentre sur tous les autres tokens de la même manière (avec un seul ensemble de poids).
*   **Limitation**: Peut avoir du mal à capturer divers types de relations ou d'informations à partir des séquences d'entrée.

### Attention multi-têtes (Multi-Head Attention)

L'attention multi-têtes, implémentée dans la classe `MultiHeadAttention`, est une extension de l'attention à tête unique. Au lieu d'effectuer un seul calcul d'attention, elle effectue plusieurs calculs d'attention (appelés 'têtes') en parallèle. Pour ce faire, les vecteurs Query, Key et Value sont d'abord projetés linéairement `num_heads` fois (ou une seule fois puis divisés), ce qui permet à chaque tête d'apprendre à se concentrer sur différentes parties de la séquence ou à capter différents types de relations. Les sorties de toutes les têtes sont ensuite concaténées et projetées linéairement pour obtenir la sortie finale.

*   **Mécanisme**: Effectue plusieurs calculs d'attention en parallèle, chacun avec ses propres projections Query/Key/Value.
*   **Vue**: Permet au modèle de se concentrer sur différentes positions à partir de différentes "têtes de représentation" et peut capturer une gamme plus riche de relations sémantiques ou syntaxiques.
*   **Avantages**: Améliore la capacité du modèle à capturer des relations complexes et diverses dans les données, en apportant un aspect d'ensemble à la capacité d'attention.

### Attention croisée (Cross-Attention)

L'attention croisée est un type d'attention multi-têtes où les Queries proviennent d'une séquence, tandis que les Keys et les Values proviennent d'une *autre* séquence. Elle est couramment utilisée dans les architectures Encodeur-Décodeur des Transformateurs (par exemple, dans la partie Décodeur). L'encodeur traite la séquence d'entrée (par exemple, une phrase source dans la traduction), et le décodeur utilise l'attention croisée pour se concentrer sur les parties pertinentes de la sortie de l'encodeur lorsqu'il génère la séquence de sortie (par exemple, une phrase cible).

*   **Mécanisme**: Les Query sont générées à partir de la séquence cible (par exemple, la sortie du décodeur), tandis que les Key et les Value sont générées à partir de la séquence source (par exemple, la sortie de l'encodeur).
*   **Vue**: Permet au modèle de mettre en correspondance des informations entre deux séquences différentes, ce qui est essentiel pour des tâches comme la traduction, la synthèse de texte ou la réponse aux questions.
*   **Exemple d'utilisation**: Dans les Transformateurs Encodeur-Décodeur, la couche de décodeur utilise l'auto-attention sur sa propre entrée masquée, puis l'attention croisée sur la sortie de l'encodeur, et enfin une couche de propagation avant.

En résumé, l'**attention à tête unique** est le bloc de construction, l'**attention multi-têtes** permet au modèle d'apprendre des relations plus riches et diverses au sein d'une seule séquence (auto-attention), et l'**attention croisée** permet au modèle de lier des informations entre deux séquences différentes.

## Récapitulatif et prochaines étapes

Nous avons parcouru les étapes clés de ce défi:

1.  **Implémentation de l'attention à tête unique**: Création d'un module `SingleHeadAttention` et validation de ses formes.
2.  **Implémentation de l'attention multi-têtes**: Extension vers `MultiHeadAttention` et intégration dans un `MultiHeadAttentionBlock` avec couches de normalisation et propagation avant, et validation des formes.
3.  **Construction d'une pile d'encodeurs personnalisés**: Assemblage des blocs d'attention en un `CustomEncoder` et encapsulé dans un `EncoderModel` pour la classification, avec validation et modification pour retourner les poids d'attention.
4.  **Préparation d'un modèle pré-entraîné**: Chargement d'un `DistilBERT` pré-entraîné de Hugging Face pour la classification de séquences, et démonstration de l'extraction de ses poids d'attention.
5.  **Explication des concepts**: Clarification des différences entre l'attention à tête unique, multi-têtes et croisée.

**Prochaines étapes suggérées:**

*   **Acquisition et prétraitement des données**: Téléchargez le jeu de données NLI et implémentez la logique de tokenisation et de création de `DataLoader` comme indiqué dans le placeholder.
*   **Entraînement et réglage fin**: Exécutez la boucle d'entraînement pour votre `EncoderModel` personnalisé et pour le `DistilBERT` (réglage fin) sur le jeu de données NLI.
*   **Analyse des cartes d'attention**: Utilisez le code conceptuel de visualisation pour tracer et interpréter les cartes d'attention des deux modèles sur des exemples spécifiques. Identifiez comment chaque modèle se concentre sur différentes parties de la prémisse et de l'hypothèse.
*   **Évaluation comparative**: Comparez les performances (précision, perte) des deux modèles sur le jeu de validation. Analysez les compromis en termes de performance, de complexité et de ressources nécessaires.